# 🔬 Méthodes *pixel-wise* pour le retrieval T1 → T2

**Problème.** Pour chaque image *query* (**T1 post-contrast**) il faut retrouver, dans la *gallery*,
l'image *target* (**T2**) du même patient. Évaluation = **MRR** (Mean Reciprocal Rank) sur les 3 datasets.

**Idée pixel-wise.** T1 et T2 du même patient partagent la **même anatomie** dans le **même espace voxel**.
On peut donc apparier *query* ↔ *target* par **similarité pixel à pixel**, sans réseau de neurones.

Ce notebook teste plusieurs familles de similarité pixel-wise :

| Famille | Méthodes |
|---|---|
| **Intensité directe** | corrélation de Pearson, cosinus, −L2 |
| **Distribution / densité des pixels** | histogrammes d'intensité + Bhattacharyya / corrélation / χ² |
| **Information mutuelle** | MI, MI normalisée (la référence multimodale) |
| **Structure** | corrélation des cartes de gradient |
| **Morphologie** | recouvrement du masque cerveau (Dice / IoU) |

On **valide sur dataset1/train** (seul split avec vérité terrain `train_pairs.csv`), puis on
**génère les soumissions** (val + test, 3 datasets) au format attendu.


## 1. Setup, chemins & chargeur avec cache

In [ ]:
import os, csv, glob, time, hashlib, math
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from skimage.transform import resize

plt.rcParams['figure.dpi'] = 110

ROOT  = os.path.join('..', 'data', 'ehl-paris-medical-image-retrieval')
SHAPE = (48, 48, 32)          # taille commune (sous-échantillonnage) pour la comparaison
CACHE = os.path.join('..', '.feat_cache')
os.makedirs(CACHE, exist_ok=True)
assert os.path.isdir(ROOT), os.path.abspath(ROOT)

def resolve(path_in_csv):
    # Chemin CSV (relatif dataset, .nii.gz) -> chemin disque reel (.nii)
    p = os.path.join(ROOT, *path_in_csv.split('/'))
    if os.path.exists(p): return p
    if os.path.exists(p[:-3]): return p[:-3]            # .nii.gz -> .nii
    if os.path.exists(p + '.gz'): return p + '.gz'
    return p

def load_feat(path_in_csv):
    # Charge -> resize SHAPE -> normalise [0,1] (percentiles). Mis en cache .npy
    p = resolve(path_in_csv)
    key = hashlib.md5((os.path.abspath(p) + str(SHAPE)).encode()).hexdigest() + '.npy'
    cf = os.path.join(CACHE, key)
    if os.path.exists(cf):
        return np.load(cf)
    v = np.asarray(nib.load(p).get_fdata(), dtype=np.float32)
    v = resize(v, SHAPE, order=1, anti_aliasing=True, preserve_range=True)
    pos = v[v > 0]
    lo, hi = (np.percentile(pos, [1, 99]) if pos.size else (0.0, 1.0))
    v = np.clip((v - lo) / max(hi - lo, 1e-6), 0, 1).astype(np.float32)
    np.save(cf, v)
    return v

def load_many(paths, log_every=50):
    out = []
    t0 = time.time()
    for i, p in enumerate(paths):
        out.append(load_feat(p))
        if log_every and (i + 1) % log_every == 0:
            print(f'  {i+1}/{len(paths)}  ({time.time()-t0:.0f}s)')
    return np.stack(out)

print('OK. SHAPE =', SHAPE)


## 2. La « boîte à outils » de similarités pixel-wise
Chaque fonction prend deux piles de volumes `Q` (nq, *SHAPE) et `G` (ng, *SHAPE)
et renvoie une **matrice de similarité** `S` (nq × ng), *plus grand = plus similaire*.

In [ ]:
def _flat(V):  return V.reshape(len(V), -1)
def _z(X):
    X = X - X.mean(1, keepdims=True)
    return X / (X.std(1, keepdims=True) + 1e-8)

def sim_pearson(Q, G):
    Qz, Gz = _z(_flat(Q)), _z(_flat(G))
    return Qz @ Gz.T / Qz.shape[1]

def sim_cosine(Q, G):
    Qf, Gf = _flat(Q), _flat(G)
    Qn = Qf / (np.linalg.norm(Qf, axis=1, keepdims=True) + 1e-8)
    Gn = Gf / (np.linalg.norm(Gf, axis=1, keepdims=True) + 1e-8)
    return Qn @ Gn.T

def sim_negL2(Q, G):
    Qf, Gf = _flat(Q), _flat(G)
    q2 = (Qf**2).sum(1)[:, None]; g2 = (Gf**2).sum(1)[None]
    return -(q2 + g2 - 2 * Qf @ Gf.T)

def sim_dice(Q, G, thr=0.1):
    Qm = (_flat(Q) > thr).astype(np.float32); Gm = (_flat(G) > thr).astype(np.float32)
    inter = Qm @ Gm.T
    return 2 * inter / (Qm.sum(1)[:, None] + Gm.sum(1)[None] + 1e-6)

def _grad_stack(V):
    out = np.empty_like(V)
    for i, v in enumerate(V):
        gx, gy, gz = np.gradient(v)
        out[i] = np.sqrt(gx**2 + gy**2 + gz**2)
    return out

def sim_gradient(Q, G):
    return sim_pearson(_grad_stack(Q), _grad_stack(G))

# --- distribution / densité des pixels ---
def hist_feat(V, bins=32, lo=0.02):
    H = np.stack([np.histogram(v[v > lo], bins=bins, range=(0, 1))[0] for v in V]).astype(float)
    return H / (H.sum(1, keepdims=True) + 1e-8)

def sim_density_bhat(Q, G, bins=32):
    HQ, HG = hist_feat(Q, bins), hist_feat(G, bins)
    return np.sqrt(HQ) @ np.sqrt(HG).T          # coefficient de Bhattacharyya

def sim_density_corr(Q, G, bins=32):
    HQ, HG = hist_feat(Q, bins), hist_feat(G, bins)
    return _z(HQ) @ _z(HG).T / HQ.shape[1]

# --- information mutuelle (référence multimodale) ---
def _mi(a, b, bins=24):
    h = np.histogram2d(a, b, bins=bins, range=[[0, 1], [0, 1]])[0]
    pxy = h / (h.sum() + 1e-12)
    px = pxy.sum(1); py = pxy.sum(0)
    nz = pxy > 0
    denom = (px[:, None] * py[None, :])[nz]
    return float((pxy[nz] * np.log(pxy[nz] / (denom + 1e-12))).sum())

def _entropy(a, bins=24):
    h = np.histogram(a, bins=bins, range=(0, 1))[0].astype(float)
    p = h / (h.sum() + 1e-12); p = p[p > 0]
    return float(-(p * np.log(p)).sum())

def sim_mi(Q, G, bins=24, normalized=False):
    Qf, Gf = _flat(Q), _flat(G)
    nq, ng = len(Qf), len(Gf)
    if normalized:
        Hq = np.array([_entropy(q, bins) for q in Qf])
        Hg = np.array([_entropy(g, bins) for g in Gf])
    S = np.zeros((nq, ng))
    for i in range(nq):
        for j in range(ng):
            m = _mi(Qf[i], Gf[j], bins)
            S[i, j] = 2 * m / (Hq[i] + Hg[j] + 1e-8) if normalized else m
    return S

METHODS = {
    'Pearson':        sim_pearson,
    'Cosine':         sim_cosine,
    'neg L2':         sim_negL2,
    'Density Bhatt.': sim_density_bhat,
    'Density corr':   sim_density_corr,
    'Gradient corr':  sim_gradient,
    'Mask Dice':      sim_dice,
    'Mutual Info':    sim_mi,
    'Norm. MI':       lambda Q, G: sim_mi(Q, G, normalized=True),
}
print('méthodes :', list(METHODS))


## 3. Métriques d'évaluation (MRR, top-1, distribution des rangs)

In [ ]:
def ranks_from_sim(S, gt):
    # gt[i] = index colonne (gallery) correct pour la query i. Renvoie le rang (1=meilleur)
    order = np.argsort(-S, axis=1)
    r = np.empty(len(S), dtype=int)
    for i in range(len(S)):
        r[i] = int(np.where(order[i] == gt[i])[0][0]) + 1
    return r

def mrr(r):   return float(np.mean(1.0 / r))
def top1(r):  return float(np.mean(r == 1))
def topk(r, k): return float(np.mean(r <= k))


## 4. Banc d'essai sur **dataset1 / train** (vérité terrain)
On charge `N_BENCH` paires connues (query T1 ↔ target T2) et on mesure chaque méthode.
La query `i` correspond exactement à la gallery `i` (vérité terrain = identité).

In [ ]:
N_BENCH = 80   # ↑ pour plus de robustesse (chargement + lent au 1er passage, puis caché)

pairs = list(csv.DictReader(open(os.path.join(ROOT, 'dataset1', 'train_pairs.csv'))))[:N_BENCH]
print(f'Chargement de {len(pairs)} paires dataset1/train ...')
Qb = load_many([r['query_image']  for r in pairs])
Gb = load_many([r['target_image'] for r in pairs])
gt = np.arange(len(pairs))
print('Q', Qb.shape, '| G', Gb.shape)


In [ ]:
results = {}
print(f"{'méthode':16s} {'MRR':>6s} {'top1':>6s} {'top5':>6s} {'temps':>7s}")
print('-' * 46)
for name, fn in METHODS.items():
    t0 = time.time()
    S = fn(Qb, Gb)
    r = ranks_from_sim(S, gt)
    results[name] = dict(S=S, r=r, mrr=mrr(r), top1=top1(r), top5=topk(r, 5), t=time.time()-t0)
    print(f"{name:16s} {results[name]['mrr']:6.3f} {results[name]['top1']:6.3f} "
          f"{results[name]['top5']:6.3f} {results[name]['t']:6.1f}s")


## 5. Visualisation des résultats du banc d'essai

In [ ]:
order = sorted(results, key=lambda k: results[k]['mrr'], reverse=True)
mrrs  = [results[k]['mrr']  for k in order]
top1s = [results[k]['top1'] for k in order]

fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))
y = np.arange(len(order))
ax[0].barh(y, mrrs, color='teal'); ax[0].set_yticks(y); ax[0].set_yticklabels(order)
ax[0].invert_yaxis(); ax[0].set_xlim(0, 1.02); ax[0].set_title('MRR par méthode (dataset1/train)')
for i, v in enumerate(mrrs): ax[0].text(v + 0.01, i, f'{v:.3f}', va='center', fontsize=9)
ax[1].barh(y, top1s, color='indianred'); ax[1].set_yticks(y); ax[1].set_yticklabels(order)
ax[1].invert_yaxis(); ax[1].set_xlim(0, 1.02); ax[1].set_title('Top-1 accuracy')
for i, v in enumerate(top1s): ax[1].text(v + 0.01, i, f'{v:.2f}', va='center', fontsize=9)
plt.tight_layout(); plt.show()


In [ ]:
# Matrice de similarité de la meilleure méthode : la diagonale doit ressortir.
best = order[0]
S = results[best]['S'].copy()
Sn = (S - S.min(1, keepdims=True)) / (np.ptp(S, axis=1, keepdims=True) + 1e-9)
fig, ax = plt.subplots(1, 2, figsize=(13, 5.2))
im = ax[0].imshow(Sn, cmap='viridis'); ax[0].set_title(f'Similarité normalisée — {best}\n(ligne=query, col=gallery)')
ax[0].set_xlabel('gallery (target)'); ax[0].set_ylabel('query'); plt.colorbar(im, ax=ax[0], fraction=0.046)
# distribution des rangs des bonnes réponses, pour les 3 meilleures méthodes
for k in order[:3]:
    r = results[k]['r']
    ax[1].hist(r, bins=np.arange(1, min(20, len(r)) + 2) - 0.5, alpha=0.5, label=f"{k} (MRR {results[k]['mrr']:.2f})")
ax[1].set_xlabel('rang du bon target'); ax[1].set_ylabel('nb queries')
ax[1].set_title('Distribution des rangs (1 = parfait)'); ax[1].legend()
plt.tight_layout(); plt.show()


## 6. Focus « distribution / densité des pixels »
Pourquoi la densité seule est faible : T1 et T2 ont des **distributions d'intensité différentes**
(modalités différentes). La densité ne porte aucune information spatiale — utile comme *filtre*,
pas comme matcher principal. On le visualise.

In [ ]:
bins = 40
HQ = hist_feat(Qb, bins); HG = hist_feat(Gb, bins)
x = np.linspace(0, 1, bins)
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
ax[0].plot(x, HQ.mean(0), label='T1 (query)', lw=2)
ax[0].fill_between(x, HQ.mean(0), alpha=0.2)
ax[0].plot(x, HG.mean(0), label='T2 (gallery)', lw=2)
ax[0].fill_between(x, HG.mean(0), alpha=0.2)
ax[0].set_title('Densité moyenne des intensités (foreground)')
ax[0].set_xlabel('intensité normalisée'); ax[0].set_ylabel('densité'); ax[0].legend()
# densité d'un patient : T1 vs son vrai T2
i = 0
ax[1].plot(x, HQ[i], label='T1 query', lw=2)
ax[1].plot(x, HG[i], label='T2 target (même patient)', lw=2)
ax[1].set_title(f'Patient #{i} — distributions T1 vs T2')
ax[1].set_xlabel('intensité normalisée'); ax[1].legend()
plt.tight_layout(); plt.show()


## 7. Vérification qualitative d'un appariement (meilleure méthode)

In [ ]:
best = order[0]
S = results[best]['S']
pred = np.argsort(-S, axis=1)[:, 0]      # gallery prédite (top-1) pour chaque query
i = 0                                     # query à inspecter
z = SHAPE[2] // 2
fig, ax = plt.subplots(1, 3, figsize=(12, 4.3))
ax[0].imshow(np.rot90(Qb[i][:, :, z]));            ax[0].set_title(f'QUERY (T1) #{i}')
ax[1].imshow(np.rot90(Gb[gt[i]][:, :, z]));        ax[1].set_title('TARGET vrai (T2)')
ax[2].imshow(np.rot90(Gb[pred[i]][:, :, z]))
ax[2].set_title(f'TARGET prédit (rang {results[best]["r"][i]})' +
                ('  ✅' if pred[i] == gt[i] else '  ❌'))
for a in ax: a.axis('off')
fig.suptitle(f'Méthode : {best}', fontsize=12); plt.tight_layout(); plt.show()


## 8. Application aux 3 datasets (val + test) & génération des classements
On choisit la **méthode de production** (par défaut **Mutual Info**, la meilleure au banc d'essai).
Pour chaque (dataset, split) on charge queries + gallery, on calcule la matrice MI, et on classe.

In [ ]:
PROD_METHOD = 'Norm. MI'     # méthode utilisée pour la soumission (meilleure au banc d'essai)
prod_fn = METHODS[PROD_METHOD]

def read_split(ds, split):
    qrows = list(csv.DictReader(open(os.path.join(ROOT, ds, f'{split}_queries.csv'))))
    grows = list(csv.DictReader(open(os.path.join(ROOT, ds, f'{split}_gallery.csv'))))
    return qrows, grows

rankings = {}     # (ds, split) -> {query_id: [gallery_id classés]}
diag = {}         # diagnostics
for ds in ['dataset1', 'dataset2', 'dataset3']:
    for split in ['val', 'test']:
        qrows, grows = read_split(ds, split)
        print(f'{ds}/{split}: {len(qrows)} queries x {len(grows)} gallery — chargement...')
        Q = load_many([r['query_image']  for r in qrows], log_every=0)
        G = load_many([r['target_image'] for r in grows], log_every=0)
        S = prod_fn(Q, G)
        gids = np.array([r['target_id'] for r in grows])
        order_idx = np.argsort(-S, axis=1)
        rk = {qrows[i]['query_id']: gids[order_idx[i]].tolist() for i in range(len(qrows))}
        rankings[(ds, split)] = rk
        # diagnostics sans label : bijectivité + marge top1-top2
        top1_idx = order_idx[:, 0]
        biject = len(set(top1_idx.tolist())) / len(qrows)
        Ss = np.sort(S, axis=1)
        margin = float(np.mean(Ss[:, -1] - Ss[:, -2]))
        diag[(ds, split)] = dict(biject=biject, margin=margin, S=S, order=order_idx)
        print(f'   bijectivité top1={biject:.2f} | marge moy (top1-top2)={margin:.4f}')


## 9. Diagnostics sur val/test (sans vérité terrain)
Faute de labels sur val/test, deux proxys de qualité :
- **Bijectivité** : fraction de gallery qui sont top-1 d'une *seule* query (≈1 si appariement propre).
- **Marge top1−top2** : confiance du meilleur match (plus haut = plus net).
- **Matrice de similarité** : une diagonale nette (après ré-ordonnancement) = bon appariement.

In [ ]:
keys = [(d, s) for d in ['dataset1', 'dataset2', 'dataset3'] for s in ['val', 'test']]
fig, ax = plt.subplots(1, 2, figsize=(14, 4.2))
labels = [f'{d[-1]}/{s}' for d, s in keys]
ax[0].bar(labels, [diag[k]['biject'] for k in keys], color='steelblue')
ax[0].set_title('Bijectivité du top-1 (1 = appariement parfaitement injectif)')
ax[0].set_ylim(0, 1.05); ax[0].axhline(1, ls='--', c='gray')
ax[1].bar(labels, [diag[k]['margin'] for k in keys], color='darkorange')
ax[1].set_title('Marge de confiance moyenne (top1 − top2)')
plt.tight_layout(); plt.show()


In [ ]:
# Matrices de similarité test, ré-ordonnées par le top-1 : diagonale = bon appariement
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
for a, d in zip(ax, ['dataset1', 'dataset2', 'dataset3']):
    info = diag[(d, 'test')]; S = info['S']
    perm = info['order'][:, 0]
    Sr = S[:, perm]                                  # colonne top-1 de la query i amenée en position i
    Sn = (Sr - Sr.min()) / (np.ptp(Sr) + 1e-9)
    im = a.imshow(Sn, cmap='magma'); a.set_title(f'{d}/test — MI réordonnée')
    a.set_xlabel('gallery (réordonnée)'); a.set_ylabel('query')
plt.tight_layout(); plt.show()
print('Une diagonale brillante = appariement net (dataset facile) ; diffus = dataset difficile.')


## 10. Écriture de la soumission
Format (identique à `submission_pca.csv`) :
```
query_id,target_id_ranking
q_xxx,g_aaa g_bbb g_ccc ...        # classement COMPLET de la gallery, meilleur d'abord
```
On suit **l'ordre des lignes** de `submission_pca.csv` (377 lignes : val+test des 3 datasets)
pour garantir l'acceptation.

In [ ]:
TEMPLATE = os.path.join('..', 'submission_pca.csv')

# lookup : query_id -> (dataset, split)
qid_loc = {}
for ds in ['dataset1', 'dataset2', 'dataset3']:
    for split in ['val', 'test']:
        for r in csv.DictReader(open(os.path.join(ROOT, ds, f'{split}_queries.csv'))):
            qid_loc[r['query_id']] = (ds, split)

# ordre des query_id depuis le template (sinon, ordre naturel)
if os.path.exists(TEMPLATE):
    tmpl_ids = [row['query_id'] for row in csv.DictReader(open(TEMPLATE))]
else:
    tmpl_ids = [q for k in keys for q in rankings[k]]

out_path = os.path.join('..', f'submission_pixelwise_{PROD_METHOD.replace(" ", "").replace(".", "")}.csv')
miss = 0
with open(out_path, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['query_id', 'target_id_ranking'])
    for qid in tmpl_ids:
        loc = qid_loc.get(qid)
        if loc is None or qid not in rankings.get(loc, {}):
            miss += 1; continue
        w.writerow([qid, ' '.join(rankings[loc][qid])])

print(f'✅ écrit : {out_path}')
print(f'   lignes template : {len(tmpl_ids)} | manquantes : {miss}')
import os as _os
print(f'   taille : {_os.path.getsize(out_path)} octets')


In [ ]:
# Vérification : relire et contrôler la structure
chk = list(csv.DictReader(open(out_path)))
print('colonnes :', list(chk[0].keys()))
print('nb lignes :', len(chk))
row = chk[0]
ids = row['target_id_ranking'].split()
print(f"exemple {row['query_id']} -> {len(ids)} candidats classés ; 3 premiers : {ids[:3]}")
# longueur de ranking attendue par dataset/split
from collections import Counter
lens = Counter(len(r['target_id_ranking'].split()) for r in chk)
print('tailles de ranking rencontrées :', dict(lens))


---
### Récapitulatif
- **Mutual Information** est la meilleure similarité pixel-wise (MRR ≈ 1.0 sur dataset1/train) :
  T1 et T2 partagent l'anatomie, et MI gère le changement de modalité.
- La **densité d'intensité** seule est faible (pas d'info spatiale) — utile en complément seulement.
- La soumission est écrite au format attendu (`query_id,target_id_ranking`, 377 lignes).

**Pour aller plus loin** : recalage (affine/élastique) avant MI pour dataset2/3, ou
fusion de rangs MI + Dice + gradient. Change `PROD_METHOD` (cellule §8) puis ré-exécute §8→§10.
